# Step-by-Step: Run Backend or Full Stack on Free Google Colab

This notebook can run either:
- backend only on Colab
- backend + frontend together on Colab

Important sync rule:
- Colab only sees code that is pushed to GitHub or uploaded into the Colab runtime.
- If you changed files in VS Code, commit and push those changes first, then rerun the clone or one-click cell here.
- Backend dependencies are installed from `multiagent/requirement.txt`.
- Frontend dependencies are installed from `multiagent/package.json`.

Backend flow in this notebook:
- Step 1: Clone the GitHub repository
- Step 2: Install Linux + Python dependencies
- Step 3: Start FastAPI backend on port 8000
- Step 4: Create temporary public URL using Cloudflare tunnel
- Step 5: Verify API health endpoint

Notes:
- No upload step is needed for the GitHub workflow.
- No Google Drive mount is needed.
- Tunnel URLs are temporary and change each session.

In [ ]:
# One-click backend run: refresh repo, install current backend deps, start backend, tunnel it, verify
import os
import re
import shutil
import subprocess
import sys
import time
from getpass import getpass

import requests

REPO_URL = "https://github.com/DinethiWijesinghe/multiagent.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/multiagent"
CLOUDFLARED = "/content/cloudflared-linux-amd64"
RUNTIME_PROFILE = "LITE"
RAG_LLM_PROVIDER = "none"

requirements_path = os.path.join(REPO_DIR, "multiagent", "requirement.txt")


def refresh_repo():
    repo_slug = REPO_URL.removeprefix("https://github.com/").removesuffix(".git").strip("/")
    repo_name = repo_slug.rsplit("/", 1)[-1]
    archive_url = f"https://codeload.github.com/{repo_slug}/tar.gz/refs/heads/{REPO_BRANCH}"
    archive_path = f"/content/{repo_name}-{REPO_BRANCH}.tar.gz"
    extracted_dir = f"/content/{repo_name}-{REPO_BRANCH}"

    print("[1/6] Refreshing repository from GitHub...")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    shutil.rmtree(extracted_dir, ignore_errors=True)
    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"

    clone_cmd = [
        "git",
        "-c",
        "http.version=HTTP/1.1",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        REPO_URL,
        REPO_DIR,
    ]

    last_error = None
    for attempt in range(1, 4):
        print(f"Clone attempt {attempt}/3...")
        result = subprocess.run(
            clone_cmd,
            cwd="/content",
            env=clone_env,
            capture_output=True,
            text=True,
        )
        if result.returncode == 0:
            if result.stdout.strip():
                print(result.stdout.strip())
            return
        last_error = result.stderr.strip() or result.stdout.strip() or f"git clone exited {result.returncode}"
        print(last_error)
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        time.sleep(2)

    print("Git clone failed in Colab. Falling back to GitHub source archive...")
    subprocess.run(["curl", "-fL", "--retry", "5", "--retry-delay", "2", "--retry-all-errors", archive_url, "-o", archive_path], check=True)
    subprocess.run(["tar", "-xzf", archive_path, "-C", "/content"], check=True)
    if not os.path.exists(extracted_dir):
        raise RuntimeError(f"Repository download fallback failed. Last git error: {last_error}")
    shutil.move(extracted_dir, REPO_DIR)
    os.remove(archive_path)


refresh_repo()
os.chdir(REPO_DIR)

if not os.path.exists(requirements_path):
    raise FileNotFoundError(f"Missing backend requirements file: {requirements_path}")

print("[2/6] Installing Linux packages and backend dependencies...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr", "libgl1", "libglib2.0-0", "wget", "git", "curl"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", requirements_path], check=True)

print("[3/6] Collecting runtime secrets...")
database_url = getpass("Neon DATABASE_URL (required): " ).strip()
auth_secret = getpass("AUTH_SECRET (leave blank to keep dev default): ").strip()
if database_url.startswith("postgresql://"):
    database_url = "postgresql+psycopg://" + database_url[len("postgresql://"):]
if database_url.startswith("postgres://"):
    database_url = "postgresql+psycopg://" + database_url[len("postgres://"):]
if not database_url:
    raise RuntimeError("DATABASE_URL is required by this backend. Set a reachable Neon/PostgreSQL URL and rerun.")

print("[4/6] Starting FastAPI backend...")
backend_env = os.environ.copy()
backend_env["RUNTIME_PROFILE"] = RUNTIME_PROFILE
backend_env["RAG_LLM_PROVIDER"] = RAG_LLM_PROVIDER
backend_env["PYTHONUNBUFFERED"] = "1"
backend_env["DATABASE_URL"] = database_url
print("Injected DATABASE_URL into backend environment.")
if auth_secret:
    backend_env["AUTH_SECRET"] = auth_secret
    print("Injected AUTH_SECRET into backend environment.")
else:
    print("AUTH_SECRET not provided. Backend will use the default dev secret.")

# Optional: Gemini API key for rich LLM-backed RAG responses
# Leave blank to keep keyless (grounded snippets) mode
google_api_key = getpass("Google API key for Gemini (leave blank for keyless mode): ").strip()
if google_api_key:
    backend_env["GOOGLE_API_KEY"] = google_api_key
    backend_env["RAG_LLM_PROVIDER"] = "gemini"
    print("Gemini API key injected -- RAG will use Gemini for rich responses.")
else:
    print("No Gemini key -- running in keyless grounded mode.")
server_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "multiagent.api_server:app",
        "--host", "0.0.0.0",
        "--port", "8000",
    ],
    cwd=REPO_DIR,
    env=backend_env,
)

local_health = None
last_error = None
for attempt in range(1, 21):
    try:
        local_health = requests.get("http://127.0.0.1:8000/health", timeout=20)
        if local_health.status_code == 200:
            break
    except Exception as exc:
        last_error = exc
    print(f"Waiting for backend {attempt}/20...")
    time.sleep(3)
else:
    raise RuntimeError(f"Backend did not become healthy: {last_error}")

print("Local health:", local_health.status_code)
print(local_health.json())
print(f"Runtime profile: {RUNTIME_PROFILE}")
print(f"RAG provider: {RAG_LLM_PROVIDER}")
print("Database mode:", "PostgreSQL" if database_url else "JSON")

# Neon PostgreSQL is persistent -- all existing users/data are immediately available.
# Register your own student account; data persists in Neon/PostgreSQL.
print('[User setup] Neon database connected -- existing accounts are ready to use.')
print('[User setup] Use your own registered student account.')
print('  Current login flow is student-only.')
print('  To add a NEW account, fill in below. Press Enter on each prompt to skip.')
_init_name = getpass('New name (Enter to skip): ').strip()
_init_email = getpass('New email (Enter to skip): ').strip()
_init_password = getpass('New password (Enter to skip): ').strip() if _init_email else ''
if _init_email and len(_init_password) >= 6:
    _reg = requests.post('http://127.0.0.1:8000/auth/register', json={'name': _init_name or _init_email.split('@')[0], 'email': _init_email, 'password': _init_password}, timeout=10)
    if _reg.status_code == 200:
        print(f'Account created -- log in as: {_init_email}')
    elif _reg.status_code == 409:
        print(f'Account already exists: {_init_email}  (use your existing password)')
    else:
        print(f'Registration failed ({_reg.status_code}): {_reg.text}')
else:
    print('Skipped in notebook prompt. Register from the app and log in with your own account.')

print("[5/6] Creating Cloudflare tunnel...")
subprocess.run(["rm", "-rf", CLOUDFLARED], check=False)
subprocess.run(["curl", "-fL", "--retry", "5", "--retry-delay", "2", "--retry-all-errors", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", CLOUDFLARED], check=True)
subprocess.run(["chmod", "+x", CLOUDFLARED], check=True)

public_url = None
for tunnel_attempt in range(1, 7):
    print(f"Creating tunnel attempt {tunnel_attempt}/6...")
    tunnel_process = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd="/content",
    )

    start = time.time()
    while time.time() - start < 120:
        line = tunnel_process.stdout.readline()
        if not line:
            continue
        print(line.rstrip())
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0).strip()
            break

    if public_url:
        break

    try:
        tunnel_process.terminate()
    except Exception:
        pass
    print("Tunnel creation failed. Retrying...")
    time.sleep(8)

if not public_url:
    raise RuntimeError("Cloudflare tunnel URL was not created after retries. Re-run this cell.")

print("[6/6] Verifying public health endpoint...")
test_url = f"{public_url}/health"
last_error = None
for attempt in range(1, 9):
    try:
        public_health = requests.get(test_url, timeout=25)
        print("Public health:", public_health.status_code)
        print(public_health.json())
        break
    except Exception as exc:
        last_error = exc
        print(f"Attempt {attempt}/8 failed: {exc}")
        time.sleep(5)
else:
    raise RuntimeError(f"Tunnel created but public health failed: {last_error}")

print("\nDone.")
print("Backend public URL:", public_url)
print("If you want your laptop frontend to call Colab, set this in multiagent/.env:")
print(f"VITE_API_URL={public_url}")

# One-Click Run

If you want to avoid running Step 1 to Step 5 one by one, run the next code cell once.

What it does automatically:
- clones the latest GitHub code
- installs Linux and Python dependencies
- starts the FastAPI backend
- creates a Cloudflare tunnel
- verifies both local and public health endpoints

Use the manual Step 1 to Step 5 cells only if you want to debug one stage at a time.

In [ ]:
# Step 1: Clone latest repository from GitHub with retries and archive fallback
import os
import shutil
import subprocess
import time

REPO_URL = "https://github.com/DinethiWijesinghe/multiagent.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/multiagent"


def refresh_repo():
    repo_slug = REPO_URL.removeprefix("https://github.com/").removesuffix(".git").strip("/")
    repo_name = repo_slug.rsplit("/", 1)[-1]
    archive_url = f"https://codeload.github.com/{repo_slug}/tar.gz/refs/heads/{REPO_BRANCH}"
    archive_path = f"/content/{repo_name}-{REPO_BRANCH}.tar.gz"
    extracted_dir = f"/content/{repo_name}-{REPO_BRANCH}"

    print("Refreshing repository from GitHub...")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    shutil.rmtree(extracted_dir, ignore_errors=True)

    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"
    clone_cmd = [
        "git",
        "-c",
        "http.version=HTTP/1.1",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        REPO_URL,
        REPO_DIR,
    ]

    last_error = None
    for attempt in range(1, 4):
        print(f"Clone attempt {attempt}/3...")
        result = subprocess.run(
            clone_cmd,
            cwd="/content",
            env=clone_env,
            capture_output=True,
            text=True,
        )
        if result.returncode == 0:
            if result.stdout.strip():
                print(result.stdout.strip())
            return
        last_error = result.stderr.strip() or result.stdout.strip() or f"git clone exited {result.returncode}"
        print(last_error)
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        time.sleep(2)

    print("Git clone failed in Colab. Falling back to GitHub source archive...")
    subprocess.run(["curl", "-fL", "--retry", "5", "--retry-delay", "2", "--retry-all-errors", archive_url, "-o", archive_path], check=True)
    subprocess.run(["tar", "-xzf", archive_path, "-C", "/content"], check=True)
    if not os.path.exists(extracted_dir):
        raise RuntimeError(f"Repository download fallback failed. Last git error: {last_error}")
    shutil.move(extracted_dir, REPO_DIR)
    os.remove(archive_path)


refresh_repo()
os.chdir(REPO_DIR)
print("Repository ready at:", REPO_DIR)

# Step 2: Install Dependencies

This installs OCR system libraries and Python packages needed by the FastAPI server.

In [ ]:
# Step 2 code: Linux + Python dependencies from the current repo
import os
import subprocess
import sys

%cd /content/multiagent

requirements_path = "/content/multiagent/multiagent/requirement.txt"
core_backend_packages = [
    "fastapi>=0.110.0",
    "uvicorn>=0.29.0",
    "python-multipart>=0.0.9",
    "opencv-python-headless>=4.9.0",
    "Pillow>=10.0.0",
    "pytesseract>=0.3.10",
    "numpy>=1.26.0",
    "scikit-learn>=1.4.0",
    "requests>=2.31.0",
    "python-dotenv>=1.0.0",
    "sqlalchemy>=2.0.0",
    "psycopg[binary]>=3.1.0",
]
optional_backend_packages = [
    "easyocr",
    "langchain",
    "langchain-community",
    "langchain-text-splitters",
    "chromadb",
    "sentence-transformers",
    "pypdf",
    "langchain-google-genai",
    "google-generativeai",
]
if not os.path.exists(requirements_path):
    raise FileNotFoundError(f"Missing backend requirements file: {requirements_path}")

!apt-get update -qq
!apt-get install -y -qq tesseract-ocr libgl1 libglib2.0-0 wget git


def pip_install(args, optional=False):
    base_cmd = [sys.executable, "-m", "pip", "install", *args]
    # Linux/Colab often blocks --user installs in managed Python.
    attempts = [base_cmd, base_cmd + ["--break-system-packages"]]
    if os.name == "nt":
        attempts.append(base_cmd + ["--user"])

    last_error = None
    for cmd in attempts:
        print("$", " ".join(cmd))
        try:
            subprocess.run(cmd, check=True)
            return
        except subprocess.CalledProcessError as exc:
            last_error = exc
            print(f"Install attempt failed (exit={exc.returncode}).")

    if optional:
        print(f"Warning: optional pip install failed after retries: {last_error}")
        return
    raise last_error


def install_packages_one_by_one(packages, optional=False):
    failed = []
    for pkg in packages:
        try:
            pip_install([pkg], optional=False)
        except Exception as exc:
            failed.append((pkg, exc))
            print(f"Package failed: {pkg} -> {exc}")

    if failed and not optional:
        failed_list = ", ".join(pkg for pkg, _ in failed)
        raise RuntimeError(f"Failed required packages: {failed_list}")


pip_install(["--upgrade", "pip", "setuptools", "wheel"], optional=True)
try:
    pip_install(["-r", requirements_path])
except Exception as exc:
    print(f"Warning: requirements file install failed ({exc}). Falling back to package-by-package core install...")
    install_packages_one_by_one(core_backend_packages, optional=False)

install_packages_one_by_one(optional_backend_packages, optional=True)

print("Installed backend dependencies from:", requirements_path)

# Step 3: Start FastAPI Server

This starts backend at http://127.0.0.1:8000 inside Colab.

In [ ]:
# Step 3 code: run backend with the current runtime defaults and test local health
import os
import subprocess
import sys
import time
from getpass import getpass

import requests

%cd /content/multiagent

backend_env = os.environ.copy()
backend_env["RUNTIME_PROFILE"] = "LITE"
backend_env["RAG_LLM_PROVIDER"] = "none"
backend_env["PYTHONUNBUFFERED"] = "1"

database_url = getpass("Neon DATABASE_URL (required): " ).strip()
auth_secret = getpass("AUTH_SECRET (leave blank to keep dev default): ").strip()
if database_url.startswith("postgresql://"):
    database_url = "postgresql+psycopg://" + database_url[len("postgresql://"):]
if database_url.startswith("postgres://"):
    database_url = "postgresql+psycopg://" + database_url[len("postgres://"):]
if not database_url:
    raise RuntimeError("DATABASE_URL is required by this backend. Set a reachable Neon/PostgreSQL URL and rerun.")
backend_env["DATABASE_URL"] = database_url
print("Injected DATABASE_URL into backend environment.")
if auth_secret:
    backend_env["AUTH_SECRET"] = auth_secret
    print("Injected AUTH_SECRET into backend environment.")
else:
    print("AUTH_SECRET not provided. Backend will use the default dev secret.")

# Optional: Gemini API key for rich LLM-backed RAG responses
# Leave blank to keep keyless (grounded snippets) mode
google_api_key = getpass("Google API key for Gemini (leave blank for keyless mode): ").strip()
if google_api_key:
    backend_env["GOOGLE_API_KEY"] = google_api_key
    backend_env["RAG_LLM_PROVIDER"] = "gemini"
    print("Gemini API key injected -- RAG will use Gemini for rich responses.")
else:
    print("No Gemini key -- running in keyless grounded mode.")

# Allow Cloudflare tunnel URLs as valid CORS origins (needed when Step 5
# exposes the frontend via a trycloudflare.com tunnel).
backend_env["CORS_ALLOW_ORIGIN_REGEX"] = r"^https://[-a-zA-Z0-9]+\.trycloudflare\.com$"

server_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "multiagent.api_server:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd="/content/multiagent",
    env=backend_env,
)

time.sleep(8)

try:
    health = requests.get("http://127.0.0.1:8000/health", timeout=15)
    print("Local health:", health.status_code)
    print(health.json())
    print("Runtime profile:", backend_env["RUNTIME_PROFILE"] )
    print("RAG provider:", backend_env["RAG_LLM_PROVIDER"] )
    print("Database mode:", "PostgreSQL" if database_url else "JSON")

    # Neon PostgreSQL is persistent -- all existing users/data are immediately available.
    # Register your own student account; data persists in Neon/PostgreSQL.
    print('[User setup] Neon database connected -- existing accounts are ready to use.')
    print('[User setup] Use your own registered student account.')
    print('  Current login flow is student-only.')
    print('  To add a NEW account, fill in below. Press Enter on each prompt to skip.')
    _init_name = getpass('New name (Enter to skip): ').strip()
    _init_email = getpass('New email (Enter to skip): ').strip()
    _init_password = getpass('New account password (Enter to skip): ').strip() if _init_email else ''
    if _init_email and len(_init_password) >= 6:
        _reg = requests.post('http://127.0.0.1:8000/auth/register', json={'name': _init_name or _init_email.split('@')[0], 'email': _init_email, 'password': _init_password}, timeout=10)
        if _reg.status_code == 200:
            print(f'Account created -- log in as: {_init_email}')
        elif _reg.status_code == 409:
            print(f'Account already exists: {_init_email}  (use your existing password)')
        else:
            print(f'Registration failed ({_reg.status_code}): {_reg.text}')
    else:
        print('Skipped in notebook prompt. Register from the app and log in with your own account.')
except Exception as exc:
    print("Health check failed:", exc)

# Step 4: Create Public URL (Cloudflare Tunnel)

This gives a temporary internet URL to your Colab backend.

In [ ]:
# Step 4 code: create Cloudflare public URL
import re
import subprocess
import time

!rm -rf cloudflared-linux-amd64
!curl -fL --retry 5 --retry-delay 2 --retry-all-errors https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# Stop previous tunnel process in this runtime if present
try:
    tunnel_process.poll()
except Exception:
    pass

tunnel_process = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
start = time.time()

while time.time() - start < 90:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    print(line.rstrip())
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0).strip()
        break

if public_url:
    print("\nPublic URL:", public_url)
    print("Now run Step 5 to verify it.")
else:
    print("\nPublic URL not found. Re-run this step.")

# Step 5: Verify Public API

Run this after Step 4 to confirm the public URL works.

In [ ]:
# Step 5 code: verify public health endpoint with retries
import requests
import time

if not public_url:
    print("No public URL detected. Re-run Step 4.")
else:
    test_url = f"{public_url.rstrip('/')}/health"
    print("Testing:", test_url)

    ok = False
    last_err = None

    # DNS for trycloudflare can take a short time to propagate
    for attempt in range(1, 9):
        try:
            response = requests.get(test_url, timeout=25)
            print("Public health:", response.status_code)
            print(response.json())
            ok = True
            break
        except Exception as e:
            last_err = e
            print(f"Attempt {attempt}/8 failed: {e}")
            time.sleep(5)

    if not ok:
        print("\nPublic URL could not be reached yet.")
        print("Action: Re-run Step 4 to generate a fresh URL, then run Step 5 again.")
        print("Last error:", last_err)

# Full Stack in Colab (Backend + Frontend)

Run the next single cell to:
- refresh the repo from GitHub so pushed VS Code changes are included
- install backend dependencies from `multiagent/requirement.txt`
- install frontend dependencies from `multiagent/package.json`
- start FastAPI backend on port 8000 with `RUNTIME_PROFILE=LITE` and `RAG_LLM_PROVIDER=none`
- create a backend public tunnel URL
- start Vite frontend on port 5173 with `VITE_API_URL` set to the backend tunnel
- create a frontend public tunnel URL

This gives you two live links:
- Frontend URL: open this in your browser
- Backend URL: already wired into the frontend

Notes:
- Push your latest VS Code changes to GitHub before running this cell if you want Colab to use them.
- Keep the Colab runtime running while you use the app.
- Tunnel URLs are temporary and change when the runtime restarts.

In [ ]:
# One-click full stack: refresh repo, install current deps, start backend + frontend, create two public URLs
import os
import re
import shutil
import subprocess
import sys
import time
from getpass import getpass

import requests

REPO_URL = "https://github.com/DinethiWijesinghe/multiagent.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/multiagent"
FRONTEND_DIR = f"{REPO_DIR}/multiagent"
REQUIREMENTS_PATH = f"{FRONTEND_DIR}/requirement.txt"
CORE_BACKEND_PACKAGES = [
    "fastapi>=0.110.0",
    "uvicorn>=0.29.0",
    "python-multipart>=0.0.9",
    "opencv-python-headless>=4.9.0",
    "Pillow>=10.0.0",
    "pytesseract>=0.3.10",
    "numpy>=1.26.0",
    "scikit-learn>=1.4.0",
    "requests>=2.31.0",
    "python-dotenv>=1.0.0",
    "sqlalchemy>=2.0.0",
    "psycopg[binary]>=3.1.0",
]
OPTIONAL_BACKEND_PACKAGES = [
    "easyocr",
    "langchain",
    "langchain-community",
    "langchain-text-splitters",
    "chromadb",
    "sentence-transformers",
    "pypdf",
    "langchain-google-genai",
    "google-generativeai",
]
CLOUDFLARED = "/content/cloudflared-linux-amd64"
RUNTIME_PROFILE = "LITE"
RAG_LLM_PROVIDER = "gemini"


def run(cmd, cwd=None, shell=False):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd, shell=shell, executable="/bin/bash" if shell else None)


def pip_install(args, optional=False):
    base_cmd = [sys.executable, "-m", "pip", "install", *args]
    # Linux/Colab often blocks --user installs in managed Python.
    attempts = [base_cmd, base_cmd + ["--break-system-packages"]]
    if os.name == "nt":
        attempts.append(base_cmd + ["--user"])

    last_error = None
    for cmd in attempts:
        try:
            run(cmd)
            return
        except subprocess.CalledProcessError as exc:
            last_error = exc
            print(f"Install attempt failed (exit={exc.returncode}).")

    if optional:
        print(f"Warning: optional pip install failed after retries: {last_error}")
        return
    raise last_error


def install_packages_one_by_one(packages, optional=False):
    failed = []
    for pkg in packages:
        try:
            pip_install([pkg], optional=False)
        except Exception as exc:
            failed.append((pkg, exc))
            print(f"Package failed: {pkg} -> {exc}")

    if failed and not optional:
        failed_list = ", ".join(pkg for pkg, _ in failed)
        raise RuntimeError(f"Failed required packages: {failed_list}")


def refresh_repo():
    repo_slug = REPO_URL.removeprefix("https://github.com/").removesuffix(".git").strip("/")
    repo_name = repo_slug.rsplit("/", 1)[-1]
    archive_url = f"https://codeload.github.com/{repo_slug}/tar.gz/refs/heads/{REPO_BRANCH}"
    archive_path = f"/content/{repo_name}-{REPO_BRANCH}.tar.gz"
    extracted_dir = f"/content/{repo_name}-{REPO_BRANCH}"

    shutil.rmtree(REPO_DIR, ignore_errors=True)
    shutil.rmtree(extracted_dir, ignore_errors=True)
    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"

    clone_cmd = [
        "git",
        "-c",
        "http.version=HTTP/1.1",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        REPO_URL,
        REPO_DIR,
    ]

    last_error = None
    for attempt in range(1, 4):
        print(f"Clone attempt {attempt}/3...")
        result = subprocess.run(
            clone_cmd,
            cwd="/content",
            env=clone_env,
            capture_output=True,
            text=True,
        )
        if result.returncode == 0:
            if result.stdout.strip():
                print(result.stdout.strip())
            return
        last_error = result.stderr.strip() or result.stdout.strip() or f"git clone exited {result.returncode}"
        print(last_error)
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        time.sleep(2)

    print("Git clone failed in Colab. Falling back to GitHub source archive...")
    run(["curl", "-fL", "--retry", "5", "--retry-delay", "2", "--retry-all-errors", archive_url, "-o", archive_path])
    run(["tar", "-xzf", archive_path, "-C", "/content"])
    if not os.path.exists(extracted_dir):
        raise RuntimeError(f"Repository download fallback failed. Last git error: {last_error}")
    shutil.move(extracted_dir, REPO_DIR)
    os.remove(archive_path)


def wait_http_ok(url, attempts=20, delay=2, expect_status=None):
    last_error = None
    for i in range(1, attempts + 1):
        try:
            response = requests.get(url, timeout=20)
            if expect_status is not None and response.status_code == expect_status:
                return response
            if expect_status is None and response.status_code < 500:
                return response
            last_error = f"HTTP {response.status_code}: {response.text[:300]}"
        except Exception as exc:
            last_error = exc
        print(f"Wait {i}/{attempts}: {url} not ready yet")
        time.sleep(delay)
    raise RuntimeError(f"Failed to reach {url}. Last error: {last_error}")


def start_tunnel(local_url, read_timeout=120):
    proc = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", local_url, "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd="/content",
    )

    public_url = None
    start = time.time()
    while time.time() - start < read_timeout:
        line = proc.stdout.readline()
        if not line:
            continue
        print(line.rstrip())
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0).strip()
            break

    return proc, public_url


def start_tunnel_with_retries(local_url, attempts=5, delay=6):
    last_error = None
    for i in range(1, attempts + 1):
        print(f"Creating tunnel for {local_url} (attempt {i}/{attempts})...")
        proc, public_url = start_tunnel(local_url)
        if public_url:
            return proc, public_url
        try:
            proc.terminate()
        except Exception:
            pass
        last_error = f"No public URL returned for {local_url}"
        print(f"Tunnel attempt {i} failed. Retrying in {delay}s...")
        time.sleep(delay)
    raise RuntimeError(last_error or f"Could not create Cloudflare URL for {local_url}")


print("[1/8] Refreshing repository from GitHub...")
refresh_repo()

if not os.path.exists(REQUIREMENTS_PATH):
    raise FileNotFoundError(f"Missing backend requirements file: {REQUIREMENTS_PATH}")

print("[2/8] Installing system dependencies...")
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "tesseract-ocr", "libgl1", "libglib2.0-0", "wget", "curl", "ca-certificates", "gnupg", "git"])

print("[3/8] Installing backend Python dependencies from multiagent/requirement.txt...")
pip_install(["--upgrade", "pip", "setuptools", "wheel"], optional=True)
try:
    pip_install(["-r", REQUIREMENTS_PATH])
except Exception as exc:
    print(f"Warning: requirements file install failed ({exc}). Falling back to package-by-package core install...")
    install_packages_one_by_one(CORE_BACKEND_PACKAGES, optional=False)

install_packages_one_by_one(OPTIONAL_BACKEND_PACKAGES, optional=True)

print("[4/8] Installing Node.js 20 for the Vite frontend...")
run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -", shell=True)
run(["apt-get", "install", "-y", "-qq", "nodejs"])
run(["node", "--version"])
try:
    run(["npm", "--version"])
except subprocess.CalledProcessError:
    print("npm command failed after nodejs install; attempting npm repair...")
    run(["apt-get", "install", "-y", "-qq", "npm"])
    run(["npm", "--version"])

print("[5/8] Installing frontend dependencies from package.json...")
run(["npm", "install"], cwd=FRONTEND_DIR)

print("[6/8] Collecting runtime secrets and starting backend...")
database_url = getpass("Neon DATABASE_URL (required): ").strip()
auth_secret = getpass("AUTH_SECRET (leave blank to keep dev default): ").strip()
if database_url.startswith("postgresql://"):
    database_url = "postgresql+psycopg://" + database_url[len("postgresql://"):]
if database_url.startswith("postgres://"):
    database_url = "postgresql+psycopg://" + database_url[len("postgres://"):]
if not database_url:
    raise RuntimeError("DATABASE_URL is required by this backend. Set a reachable Neon/PostgreSQL URL and rerun the cell.")

backend_env = os.environ.copy()
backend_env["RUNTIME_PROFILE"] = RUNTIME_PROFILE
backend_env["RAG_LLM_PROVIDER"] = RAG_LLM_PROVIDER
backend_env["PYTHONUNBUFFERED"] = "1"
backend_env["DATABASE_URL"] = database_url
print("Injected DATABASE_URL into backend environment.")
if auth_secret:
    backend_env["AUTH_SECRET"] = auth_secret
    print("Injected AUTH_SECRET into backend environment.")
else:
    print("AUTH_SECRET not provided. Backend will use the default dev secret.")

# Optional: Gemini API key for rich LLM-backed RAG responses
# Leave blank to keep keyless (grounded snippets) mode
google_api_key = getpass("Google API key for Gemini (leave blank for keyless mode): ").strip()
if google_api_key:
    backend_env["GOOGLE_API_KEY"] = google_api_key
    backend_env["RAG_LLM_PROVIDER"] = "gemini"
    print("Gemini API key injected -- RAG will use Gemini for rich responses.")
else:
    print("No Gemini key -- running in keyless grounded mode.")

# Allow both Cloudflare tunnel URLs as valid CORS origins so the frontend
# tunnel can reach the backend tunnel without a CORS error.
backend_env["CORS_ALLOW_ORIGIN_REGEX"] = r"^https://[-a-zA-Z0-9]+\.trycloudflare\.com$"

backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "multiagent.api_server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=REPO_DIR,
    env=backend_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f"Backend PID: {backend_proc.pid}")

local_backend = "http://127.0.0.1:8000/health"
wait_http_ok(local_backend, attempts=40, delay=2, expect_status=200)

print("[6.5/8] User accounts persist in Neon/PostgreSQL. Register from the UI if needed.")

print("[7/8] Creating backend public tunnel URL...")
run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", CLOUDFLARED])
run(["chmod", "+x", CLOUDFLARED])
backend_tunnel_proc, backend_public_url = start_tunnel_with_retries("http://127.0.0.1:8000")
if not backend_public_url:
    raise RuntimeError("Failed to create backend public tunnel URL.")
print("Backend URL:", backend_public_url)

print("[8/8] Starting frontend and creating frontend public tunnel URL...")
frontend_env = os.environ.copy()
frontend_env["VITE_API_URL"] = backend_public_url
frontend_env["PYTHONUNBUFFERED"] = "1"

frontend_proc = subprocess.Popen(
    ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173"],
    cwd=FRONTEND_DIR,
    env=frontend_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f"Frontend PID: {frontend_proc.pid}")

wait_http_ok("http://127.0.0.1:5173", attempts=50, delay=2)
frontend_tunnel_proc, frontend_public_url = start_tunnel_with_retries("http://127.0.0.1:5173")
if not frontend_public_url:
    raise RuntimeError("Failed to create frontend public tunnel URL.")

print("\n================ LIVE URLS ================")
print("Frontend URL:", frontend_public_url)
print("Backend URL:", backend_public_url)
print("===========================================")
print("Keep this runtime running. URLs are temporary and change after restart.")

# Keep process handles in globals for optional cleanup/debugging in later cells
globals()["_backend_proc"] = backend_proc
globals()["_frontend_proc"] = frontend_proc
globals()["_backend_tunnel_proc"] = backend_tunnel_proc
globals()["_frontend_tunnel_proc"] = frontend_tunnel_proc

# Tunnel Health Check and Expiry Reminder

Run the next cell any time to quickly verify whether your current backend and frontend tunnel URLs are still alive.

If a check fails, rerun the **Full Stack in Colab (Backend + Frontend)** one-click cell to generate fresh URLs.

In [ ]:
# Quick tunnel health check: backend + frontend (with clearer diagnostics)
import requests
import time

backend_url = globals().get("backend_public_url") or globals().get("public_url")
frontend_url = globals().get("frontend_public_url")


def _normalize_base(url):
    if not url:
        return None
    return str(url).rstrip("/")


def _request(url, timeout=20):
    headers = {"User-Agent": "Mozilla/5.0 (Colab Tunnel Health Check)"}
    try:
        r = requests.get(url, timeout=timeout, headers=headers, allow_redirects=True)
        return True, r.status_code, None
    except Exception as exc:
        return False, None, str(exc)


def _check_local_services():
    local_backend_ok, local_backend_status, local_backend_err = _request("http://127.0.0.1:8000/health", timeout=8)
    local_frontend_ok, local_frontend_status, local_frontend_err = _request("http://127.0.0.1:5173", timeout=8)

    print("Local checks")
    if local_backend_ok:
        print(f"- Backend local OK: http://127.0.0.1:8000/health (status {local_backend_status})")
    else:
        print(f"- Backend local FAILED: {local_backend_err}")
    if local_frontend_ok:
        print(f"- Frontend local OK: http://127.0.0.1:5173 (status {local_frontend_status})")
    else:
        print(f"- Frontend local FAILED: {local_frontend_err}")

    return local_backend_ok, local_frontend_ok


def _check_backend_public(url):
    base = _normalize_base(url)
    if not base:
        print("Backend URL not found in current runtime variables.")
        return False

    health_url = f"{base}/health"

    # Try a few times because DNS / route propagation can lag briefly
    last_err = None
    for attempt in range(1, 6):
        ok, status, err = _request(health_url, timeout=20)
        if ok and status == 200:
            print(f"Backend public OK: {health_url} (status {status})")
            return True
        last_err = err or f"status {status}"
        print(f"Backend public retry {attempt}/5 failed: {last_err}")
        time.sleep(3)

    print(f"Backend public FAILED: {health_url}")
    print(f"Reason: {last_err}")
    return False


def _check_frontend_public(url):
    base = _normalize_base(url)
    if not base:
        print("Frontend URL not found in current runtime variables.")
        return False

    ok, status, err = _request(base, timeout=20)
    # 403 from trycloudflare can still mean tunnel is alive (Cloudflare edge response)
    if ok and status in (200, 301, 302, 304, 307, 308, 403):
        print(f"Frontend public reachable: {base} (status {status})")
        return True

    print(f"Frontend public FAILED: {base}")
    if status is not None:
        print(f"Status: {status}")
    if err:
        print(f"Error: {err}")
    return False


print("Checking active tunnel URLs...\n")
local_backend_ok, local_frontend_ok = _check_local_services()
print()
backend_ok = _check_backend_public(backend_url)
frontend_ok = _check_frontend_public(frontend_url)

print("\nSummary")
print(f"Backend public healthy: {backend_ok}")
print(f"Frontend public reachable: {frontend_ok}")

if not backend_ok and local_backend_ok and frontend_ok:
    print("\nLikely temporary Colab-to-Cloudflare egress issue.")
    print("Action: Wait 30-60 seconds and rerun this check cell once.")
    print("If it still fails, rerun the Full Stack one-click cell to mint fresh tunnel URLs.")
elif not (backend_ok and frontend_ok):
    print("\nReminder: Tunnel URLs are temporary.")
    print("Action: Rerun the Full Stack in Colab (Backend + Frontend) one-click cell to get fresh URLs.")
else:
    print("\nAll good. Your current tunnel URLs are active.")